In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

pd.set_option('display.max_columns', 200)
pd.set_option('display.max_rows', 200)

plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

data_dir = Path('./jd_marts_output')

sku_master = pd.read_csv(data_dir / 'sku_master.csv', low_memory=False)
user_master = pd.read_csv(data_dir / 'user_master.csv', low_memory=False)

order_item_fact = pd.read_csv(
    data_dir / 'order_item_fact.csv',
    parse_dates=['order_time', 'order_date', 'activate_date', 'deactivate_date'],
    low_memory=False
)

click_intent_fact = pd.read_csv(
    data_dir / 'click_intent_fact.csv',
    parse_dates=['first_click_ts', 'last_click_ts', 'click_date'],
    low_memory=False
)

print('sku_master:', sku_master.shape)
print('user_master:', user_master.shape)
print('order_item_fact:', order_item_fact.shape)
print('click_intent_fact:', click_intent_fact.shape)

sku_master: (31868, 18)
user_master: (457298, 16)
order_item_fact: (549989, 59)
click_intent_fact: (3026685, 20)


In [4]:
user_master['user_id'] = user_master['user_id'].astype('string').str.strip()

valid_user_mask = (
    user_master['user_id'].notna() &
    (user_master['user_id'] != '') &
    (user_master['user_id'] != '-') &
    (user_master['user_id'].str.lower() != 'nan')
)

valid_user_set = set(user_master.loc[valid_user_mask, 'user_id'])

print('有效 user_id 数量:', len(valid_user_set))
print('是否包含 "-" ?', '-' in valid_user_set)

有效 user_id 数量: 457298
是否包含 "-" ? False


In [5]:
click_sample = click_intent_fact[
    [
        'user_id', 'sku_id', 'channel', 'click_date',
        'first_click_ts', 'last_click_ts', 'click_cnt',
        'hour', 'dow', 'is_weekend', 'hour_bucket'
    ]
].copy()

click_sample['user_id'] = click_sample['user_id'].astype('string').str.strip()

click_sample_valid = click_sample[
    click_sample['user_id'].isin(valid_user_set)
].copy()

print('原始 click_sample 行数:', len(click_sample))
print('过滤后 click_sample_valid 行数:', len(click_sample_valid))
print('过滤后是否还包含 "-" :', (click_sample_valid["user_id"] == "-").any())

原始 click_sample 行数: 3026685
过滤后 click_sample_valid 行数: 871467
过滤后是否还包含 "-" : False


In [6]:
user_day_stats = (
    click_sample_valid
    .groupby(['user_id', 'click_date'], as_index=False)
    .agg(
        user_day_total_clicks=('click_cnt', 'sum'),
        user_day_distinct_skus=('sku_id', 'nunique'),
        user_day_distinct_channels=('channel', 'nunique')
    )
)

print(user_day_stats.shape)
user_day_stats.head(10)

(274502, 5)


,user_id,click_date,user_day_total_clicks,user_day_distinct_skus,user_day_distinct_channels
0,0001bbdc89,2018-03-04,2,2,1
1,0001bbdc89,2018-03-05,4,2,1
2,0001f75444,2018-03-01,9,4,1
3,0001f75444,2018-03-04,15,6,1
4,0001f75444,2018-03-05,1,1,1
5,0001f75444,2018-03-08,1,1,1
6,00020f8411,2018-03-09,4,3,1
7,00039f650a,2018-03-04,4,3,1
8,00039f650a,2018-03-08,4,2,1
9,00047a0b67,2018-03-01,5,2,1


In [7]:
click_sample_valid = click_sample_valid.merge(
    user_day_stats,
    on=['user_id', 'click_date'],
    how='left',
    validate='many_to_one'
)

click_sample_valid.head(20)

,user_id,sku_id,channel,click_date,first_click_ts,last_click_ts,click_cnt,hour,dow,is_weekend,hour_bucket,user_day_total_clicks,user_day_distinct_skus,user_day_distinct_channels
0,0001bbdc89,5f58bfd286,app,2018-03-05,2018-03-05 09:05:01,2018-03-05 15:44:23,3,9.0,0.0,0,morning,4,2,1
1,0001bbdc89,ac0cd64708,app,2018-03-05,2018-03-05 09:04:33,2018-03-05 09:04:33,1,9.0,0.0,0,morning,4,2,1
2,0001bbdc89,f255daf3ab,app,2018-03-04,2018-03-04 20:58:46,2018-03-04 20:58:46,1,20.0,6.0,1,evening,2,2,1
3,0001bbdc89,fc728c8ac1,app,2018-03-04,2018-03-04 20:52:58,2018-03-04 20:52:58,1,20.0,6.0,1,evening,2,2,1
4,0001f75444,2189b70880,app,2018-03-04,2018-03-04 21:07:26,2018-03-04 21:07:26,1,21.0,6.0,1,evening,15,6,1
5,0001f75444,4d380cf119,app,2018-03-04,2018-03-04 19:16:31,2018-03-04 21:29:03,5,19.0,6.0,1,evening,15,6,1
6,0001f75444,a0e49f9966,app,2018-03-01,2018-03-01 15:33:29,2018-03-01 17:40:16,2,17.0,3.0,0,afternoon,9,4,1
7,0001f75444,a0e49f9966,app,2018-03-04,2018-03-04 19:19:03,2018-03-04 19:19:03,1,19.0,6.0,1,evening,15,6,1
8,0001f75444,b8c182c74f,app,2018-03-01,2018-03-01 15:18:09,2018-03-01 21:25:47,2,15.0,3.0,0,afternoon,9,4,1
9,0001f75444,b8c182c74f,app,2018-03-04,2018-03-04 19:31:19,2018-03-04 21:03:06,3,21.0,6.0,1,evening,15,6,1


In [8]:
tmp = click_sample_valid.iloc[0]
u = tmp['user_id']
d = tmp['click_date']

check_detail = click_sample_valid[
    (click_sample_valid['user_id'] == u) &
    (click_sample_valid['click_date'] == d)
].copy()

print('user_id =', u)
print('click_date =', d)
print('click_cnt 求和 =', check_detail['click_cnt'].sum())
print('不同 sku 数 =', check_detail['sku_id'].nunique())
print('不同 channel 数 =', check_detail['channel'].nunique())

print('表中 user_day_total_clicks =', check_detail['user_day_total_clicks'].iloc[0])
print('表中 user_day_distinct_skus =', check_detail['user_day_distinct_skus'].iloc[0])
print('表中 user_day_distinct_channels =', check_detail['user_day_distinct_channels'].iloc[0])

user_id = 0001bbdc89
click_date = 2018-03-05 00:00:00
click_cnt 求和 = 4
不同 sku 数 = 2
不同 channel 数 = 1
表中 user_day_total_clicks = 4
表中 user_day_distinct_skus = 2
表中 user_day_distinct_channels = 1


In [9]:
click_sample_valid = click_sample_valid.sort_values(
    ['user_id', 'sku_id', 'last_click_ts']
).reset_index(drop=True)

click_sample_valid['user_sku_click_days_so_far'] = (
    click_sample_valid.groupby(['user_id', 'sku_id']).cumcount() + 1
)

click_sample_valid['user_sku_clicks_so_far'] = (
    click_sample_valid.groupby(['user_id', 'sku_id'])['click_cnt'].cumsum()
)

click_sample_valid['repeat_same_sku_flag'] = (
    click_sample_valid['user_sku_click_days_so_far'] >= 2
).astype('int8')

click_sample_valid['same_day_multi_click_flag'] = (
    click_sample_valid['click_cnt'] >= 2
).astype('int8')

click_sample_valid.head()

,user_id,sku_id,channel,click_date,first_click_ts,last_click_ts,click_cnt,hour,dow,is_weekend,hour_bucket,user_day_total_clicks,user_day_distinct_skus,user_day_distinct_channels,user_sku_click_days_so_far,user_sku_clicks_so_far,repeat_same_sku_flag,same_day_multi_click_flag
0,0001bbdc89,5f58bfd286,app,2018-03-05,2018-03-05 09:05:01,2018-03-05 15:44:23,3,9.0,0.0,0,morning,4,2,1,1,3,0,1
1,0001bbdc89,ac0cd64708,app,2018-03-05,2018-03-05 09:04:33,2018-03-05 09:04:33,1,9.0,0.0,0,morning,4,2,1,1,1,0,0
2,0001bbdc89,f255daf3ab,app,2018-03-04,2018-03-04 20:58:46,2018-03-04 20:58:46,1,20.0,6.0,1,evening,2,2,1,1,1,0,0
3,0001bbdc89,fc728c8ac1,app,2018-03-04,2018-03-04 20:52:58,2018-03-04 20:52:58,1,20.0,6.0,1,evening,2,2,1,1,1,0,0
4,0001f75444,2189b70880,app,2018-03-04,2018-03-04 21:07:26,2018-03-04 21:07:26,1,21.0,6.0,1,evening,15,6,1,1,1,0,0


In [37]:
click_sample_valid['same_sku_click_share_in_day'] = np.where(
    click_sample_valid['user_day_total_clicks'] == 0,
    np.nan,
    click_sample_valid['click_cnt'] / click_sample_valid['user_day_total_clicks']
)

click_sample_valid['sku_focus_flag'] = (
    click_sample_valid['same_sku_click_share_in_day'] >= 0.5
).astype('int8')

click_sample_valid.head(20)

,user_id,sku_id,channel,click_date,first_click_ts,last_click_ts,click_cnt,hour,dow,is_weekend,hour_bucket,user_day_total_clicks,user_day_distinct_skus,user_day_distinct_channels,user_sku_click_days_so_far,user_sku_clicks_so_far,repeat_same_sku_flag,same_day_multi_click_flag,same_sku_click_share_in_day,sku_focus_flag,is_plus,purchase_power,city_level,gender,age,user_level,type,brand_id,any_attr_missing,any_attr_placeholder,brand_sku_cnt,user_sku_key
0,0001bbdc89,5f58bfd286,app,2018-03-05,2018-03-05 09:05:01,2018-03-05 15:44:23,3,9.0,0.0,0,morning,4,2,1,1,3,0,1,0.750000,1,0,2,2,F,16-25,2,1,9b0d3a5fc6,0,1,831,0001bbdc89||5f58bfd286
1,0001bbdc89,ac0cd64708,app,2018-03-05,2018-03-05 09:04:33,2018-03-05 09:04:33,1,9.0,0.0,0,morning,4,2,1,1,1,0,0,0.250000,0,0,2,2,F,16-25,2,1,9b0d3a5fc6,0,0,831,0001bbdc89||ac0cd64708
2,0001bbdc89,f255daf3ab,app,2018-03-04,2018-03-04 20:58:46,2018-03-04 20:58:46,1,20.0,6.0,1,evening,2,2,1,1,1,0,0,0.500000,1,0,2,2,F,16-25,2,1,4efb032b5a,0,0,539,0001bbdc89||f255daf3ab
3,0001bbdc89,fc728c8ac1,app,2018-03-04,2018-03-04 20:52:58,2018-03-04 20:52:58,1,20.0,6.0,1,evening,2,2,1,1,1,0,0,0.500000,1,0,2,2,F,16-25,2,1,4efb032b5a,0,0,539,0001bbdc89||fc728c8ac1
4,0001f75444,2189b70880,app,2018-03-04,2018-03-04 21:07:26,2018-03-04 21:07:26,1,21.0,6.0,1,evening,15,6,1,1,1,0,0,0.066667,0,0,3,3,F,>=56,3,1,7cc01be867,0,1,253,0001f75444||2189b70880
5,0001f75444,4d380cf119,app,2018-03-04,2018-03-04 19:16:31,2018-03-04 21:29:03,5,19.0,6.0,1,evening,15,6,1,1,5,0,1,0.333333,0,0,3,3,F,>=56,3,1,7cc01be867,0,0,253,0001f75444||4d380cf119
6,0001f75444,a0e49f9966,app,2018-03-01,2018-03-01 15:33:29,2018-03-01 17:40:16,2,17.0,3.0,0,afternoon,9,4,1,1,2,0,1,0.222222,0,0,3,3,F,>=56,3,1,7cc01be867,0,1,253,0001f75444||a0e49f9966
7,0001f75444,a0e49f9966,app,2018-03-04,2018-03-04 19:19:03,2018-03-04 19:19:03,1,19.0,6.0,1,evening,15,6,1,2,3,1,0,0.066667,0,0,3,3,F,>=56,3,1,7cc01be867,0,1,253,0001f75444||a0e49f9966
8,0001f75444,b8c182c74f,app,2018-03-01,2018-03-01 15:18:09,2018-03-01 21:25:47,2,15.0,3.0,0,afternoon,9,4,1,1,2,0,1,0.222222,0,0,3,3,F,>=56,3,1,7cc01be867,0,1,253,0001f75444||b8c182c74f
9,0001f75444,b8c182c74f,app,2018-03-04,2018-03-04 19:31:19,2018-03-04 21:03:06,3,21.0,6.0,1,evening,15,6,1,2,5,1,1,0.200000,0,0,3,3,F,>=56,3,1,7cc01be867,0,1,253,0001f75444||b8c182c74f


In [11]:
user_features = user_master[
    ['user_id', 'is_plus', 'purchase_power', 'city_level', 'gender', 'age', 'user_level']
].copy()

sku_features = sku_master[
    ['sku_id', 'type', 'brand_id', 'any_attr_missing', 'any_attr_placeholder', 'brand_sku_cnt']
].copy()

click_sample_valid = (
    click_sample_valid
    .merge(user_features, on='user_id', how='left')
    .merge(sku_features, on='sku_id', how='left')
)

click_sample_valid.head()

,user_id,sku_id,channel,click_date,first_click_ts,last_click_ts,click_cnt,hour,dow,is_weekend,hour_bucket,user_day_total_clicks,user_day_distinct_skus,user_day_distinct_channels,user_sku_click_days_so_far,user_sku_clicks_so_far,repeat_same_sku_flag,same_day_multi_click_flag,same_sku_click_share_in_day,sku_focus_flag,is_plus,purchase_power,city_level,gender,age,user_level,type,brand_id,any_attr_missing,any_attr_placeholder,brand_sku_cnt
0,0001bbdc89,5f58bfd286,app,2018-03-05,2018-03-05 09:05:01,2018-03-05 15:44:23,3,9.0,0.0,0,morning,4,2,1,1,3,0,1,0.750000,1,0,2,2,F,16-25,2,1,9b0d3a5fc6,0,1,831
1,0001bbdc89,ac0cd64708,app,2018-03-05,2018-03-05 09:04:33,2018-03-05 09:04:33,1,9.0,0.0,0,morning,4,2,1,1,1,0,0,0.250000,0,0,2,2,F,16-25,2,1,9b0d3a5fc6,0,0,831
2,0001bbdc89,f255daf3ab,app,2018-03-04,2018-03-04 20:58:46,2018-03-04 20:58:46,1,20.0,6.0,1,evening,2,2,1,1,1,0,0,0.500000,1,0,2,2,F,16-25,2,1,4efb032b5a,0,0,539
3,0001bbdc89,fc728c8ac1,app,2018-03-04,2018-03-04 20:52:58,2018-03-04 20:52:58,1,20.0,6.0,1,evening,2,2,1,1,1,0,0,0.500000,1,0,2,2,F,16-25,2,1,4efb032b5a,0,0,539
4,0001f75444,2189b70880,app,2018-03-04,2018-03-04 21:07:26,2018-03-04 21:07:26,1,21.0,6.0,1,evening,15,6,1,1,1,0,0,0.066667,0,0,3,3,F,>=56,3,1,7cc01be867,0,1,253


In [12]:
order_events = (
    order_item_fact
    .groupby(['user_id', 'sku_id', 'order_id'], as_index=False)
    .agg(
        order_time=('order_time', 'min'),
        order_date=('order_date', 'min'),
        order_gmv=('gmv', 'sum'),
        order_qty=('quantity', 'sum')
    )
    .sort_values(['user_id', 'sku_id', 'order_time'])
    .reset_index(drop=True)
)

order_events.head()

,user_id,sku_id,order_id,order_time,order_date,order_gmv,order_qty
0,000089d6a6,e99eb7d131,6fb419a6de,2018-03-14 14:50:39,2018-03-14,215.0,1
1,0000babd1f,7185ef8e8c,6f20820bed,2018-03-22 14:40:10,2018-03-22,39.0,1
2,0000bc018b,fa823767ca,ebbf0f8a69,2018-03-30 15:58:03,2018-03-30,79.0,1
3,0000d0e5ab,2523d051fd,e8081938a6,2018-03-28 11:48:21,2018-03-28,228.0,1
4,0000dce472,f0e625dda4,3f68275300,2018-03-18 23:27:45,2018-03-18,111.5,1


In [13]:
click_sample_valid['user_id'] = click_sample_valid['user_id'].astype('string').str.strip()
click_sample_valid['sku_id'] = click_sample_valid['sku_id'].astype('string').str.strip()

click_sample_valid = click_sample_valid.dropna(subset=['user_id', 'sku_id', 'last_click_ts']).copy()

click_sample_valid = click_sample_valid[
    (click_sample_valid['user_id'] != '') &
    (click_sample_valid['sku_id'] != '')
].copy()

click_sample_valid['user_sku_key'] = (
    click_sample_valid['user_id'] + '||' + click_sample_valid['sku_id']
)

print('click_sample_valid 行数:', len(click_sample_valid))
print('click_sample_valid 中 last_click_ts 空值数:', click_sample_valid['last_click_ts'].isna().sum())
print('click_sample_valid 中 user_sku_key 空值数:', click_sample_valid['user_sku_key'].isna().sum())

click_sample_valid 行数: 871467
click_sample_valid 中 last_click_ts 空值数: 0
click_sample_valid 中 user_sku_key 空值数: 0


In [14]:
order_events['user_id'] = order_events['user_id'].astype('string').str.strip()
order_events['sku_id'] = order_events['sku_id'].astype('string').str.strip()

order_events = order_events.dropna(subset=['user_id', 'sku_id', 'order_time']).copy()

order_events = order_events[
    (order_events['user_id'] != '') &
    (order_events['sku_id'] != '')
].copy()

order_events['user_sku_key'] = (
    order_events['user_id'] + '||' + order_events['sku_id']
)

print('order_events 行数:', len(order_events))
print('order_events 中 order_time 空值数:', order_events['order_time'].isna().sum())
print('order_events 中 user_sku_key 空值数:', order_events['user_sku_key'].isna().sum())

order_events 行数: 541097
order_events 中 order_time 空值数: 0
order_events 中 user_sku_key 空值数: 0


In [15]:
click_for_label = click_sample_valid.sort_values(
    ['user_sku_key', 'last_click_ts']
).reset_index(drop=True)

orders_for_label = order_events.sort_values(
    ['user_sku_key', 'order_time']
).reset_index(drop=True)

In [16]:
print('left last_click_ts 空值数:', click_for_label['last_click_ts'].isna().sum())
print('right order_time 空值数:', orders_for_label['order_time'].isna().sum())
print('left user_sku_key 空值数:', click_for_label['user_sku_key'].isna().sum())
print('right user_sku_key 空值数:', orders_for_label['user_sku_key'].isna().sum())

left last_click_ts 空值数: 0
right order_time 空值数: 0
left user_sku_key 空值数: 0
right user_sku_key 空值数: 0


In [17]:
click_for_label = click_sample_valid.sort_values(
    ['last_click_ts', 'user_sku_key'],
    kind='mergesort'
).reset_index(drop=True)

orders_for_label = order_events.sort_values(
    ['order_time', 'user_sku_key'],
    kind='mergesort'
).reset_index(drop=True)

In [18]:
print('left 时间是否全局递增:', click_for_label['last_click_ts'].is_monotonic_increasing)
print('right 时间是否全局递增:', orders_for_label['order_time'].is_monotonic_increasing)

left 时间是否全局递增: True
right 时间是否全局递增: True


In [39]:
orders_for_merge = orders_for_label[
    ['user_sku_key', 'order_time', 'order_id', 'order_gmv', 'order_qty']
].copy()

labeled_sample = pd.merge_asof(
    click_for_label,
    orders_for_merge,
    left_on='last_click_ts',
    right_on='order_time',
    by='user_sku_key',
    direction='forward',
    allow_exact_matches=False
)

labeled_sample['days_to_purchase'] = (
    labeled_sample['order_time'] - labeled_sample['last_click_ts']
).dt.total_seconds() / (3600 * 24)

labeled_sample[
    ['user_id', 'sku_id', 'channel', 'last_click_ts', 'order_time', 'days_to_purchase']
].head(100)

,user_id,sku_id,channel,last_click_ts,order_time,days_to_purchase
0,7a89b29ba5,a0e49f9966,app,2018-02-28 23:59:01,2018-03-02 10:08:04,1.422951
1,ba189a22b7,2f268cf558,app,2018-02-28 23:59:02,NaT,NaN
2,e03f8c6d4e,6a0f1004bb,app,2018-02-28 23:59:02,NaT,NaN
3,7093c5f021,861f71f9ad,app,2018-02-28 23:59:04,NaT,NaN
4,7492b76bdf,d3e31fdd6e,app,2018-02-28 23:59:04,2018-03-01 00:15:48,0.011620
5,27c9fdcbd2,fbce41fd82,app,2018-02-28 23:59:05,2018-03-01 00:07:33,0.005880
6,b5f84d46c9,38676d45f5,app,2018-02-28 23:59:06,2018-03-01 10:12:55,0.426262
7,352808da84,2523d051fd,app,2018-02-28 23:59:07,NaT,NaN
8,6095b44680,ac61f4e10e,app,2018-02-28 23:59:07,NaT,NaN
9,6095b44680,bd27bf28a1,app,2018-02-28 23:59:07,NaT,NaN


In [20]:
print('总样本数:', len(labeled_sample))
print('匹配到后续购买的样本数:', labeled_sample['order_time'].notna().sum())
print('匹配率:', labeled_sample['order_time'].notna().mean())

总样本数: 871467
匹配到后续购买的样本数: 131403
匹配率: 0.1507836785558145


In [21]:
labeled_sample['days_to_purchase'].describe()

count    131403.000000
mean          2.141019
std           4.971658
min           0.000012
25%           0.002222
50%           0.017812
75%           1.370972
max          30.766493
Name: days_to_purchase, dtype: float64

In [22]:
print('days_to_purchase < 0 的样本数:', (labeled_sample['days_to_purchase'] < 0).sum())

days_to_purchase < 0 的样本数: 0


In [23]:
labeled_sample['label_1d'] = (
    (labeled_sample['days_to_purchase'] >= 0) &
    (labeled_sample['days_to_purchase'] <= 1)
).astype('int8')

labeled_sample['label_3d'] = (
    (labeled_sample['days_to_purchase'] >= 0) &
    (labeled_sample['days_to_purchase'] <= 3)
).astype('int8')

labeled_sample['label_7d'] = (
    (labeled_sample['days_to_purchase'] >= 0) &
    (labeled_sample['days_to_purchase'] <= 7)
).astype('int8')

print('1天内购买率:', labeled_sample['label_1d'].mean())
print('3天内购买率:', labeled_sample['label_3d'].mean())
print('7天内购买率:', labeled_sample['label_7d'].mean())

1天内购买率: 0.10807752904011282
3天内购买率: 0.12483433107621976
7天内购买率: 0.13624268044573115


In [24]:
max_click_ts = labeled_sample['last_click_ts'].max()
cutoff_7d = max_click_ts - pd.Timedelta(days=7)

sample_7d = labeled_sample[
    labeled_sample['last_click_ts'] <= cutoff_7d
].copy()

print('原始样本数:', len(labeled_sample))
print('7天标签有效样本数:', len(sample_7d))
print('7天购买率:', sample_7d['label_7d'].mean())

原始样本数: 871467
7天标签有效样本数: 213050
7天购买率: 0.13928185871861065


In [25]:
channel_summary = (
    sample_7d
    .groupby('channel', as_index=False)
    .agg(
        click_samples=('user_id', 'size'),
        users=('user_id', 'nunique'),
        skus=('sku_id', 'nunique'),
        conv_1d=('label_1d', 'mean'),
        conv_3d=('label_3d', 'mean'),
        conv_7d=('label_7d', 'mean'),
        avg_click_cnt=('click_cnt', 'mean'),
        avg_user_day_distinct_skus=('user_day_distinct_skus', 'mean')
    )
    .sort_values('click_samples', ascending=False)
)

display(channel_summary)

,channel,click_samples,users,skus,conv_1d,conv_3d,conv_7d,avg_click_cnt,avg_user_day_distinct_skus
0,app,178257,42946,6800,0.103940,0.120455,0.134912,2.514970,8.280752
4,wechat,17684,4517,3049,0.099468,0.113606,0.127064,2.150645,8.549310
3,pc,10775,2998,1658,0.163898,0.177912,0.194988,1.875638,8.081392
1,mobile,3794,1450,838,0.192673,0.212968,0.226410,1.849499,6.792040
2,others,2540,904,1046,0.139764,0.153937,0.164567,1.822835,9.286220


In [26]:
sample_7d['click_cnt_bucket'] = pd.cut(
    sample_7d['click_cnt'],
    bins=[0, 1, 2, 3, 5, 10, 999999],
    labels=['1', '2', '3', '4-5', '6-10', '10+']
)

click_bucket_summary = (
    sample_7d
    .groupby('click_cnt_bucket', observed=True, as_index=False)
    .agg(
        samples=('user_id', 'size'),
        conv_7d=('label_7d', 'mean')
    )
)

display(click_bucket_summary)

,click_cnt_bucket,samples,conv_7d
0,1,115309,0.079048
1,2,39986,0.151578
2,3,19922,0.213483
3,4-5,19051,0.248123
4,6-10,13916,0.297643
5,10+,4866,0.282778


In [27]:
focus_summary = (
    sample_7d
    .groupby('sku_focus_flag', as_index=False)
    .agg(
        samples=('user_id', 'size'),
        conv_7d=('label_7d', 'mean'),
        avg_click_cnt=('click_cnt', 'mean'),
        avg_distinct_skus=('user_day_distinct_skus', 'mean')
    )
)

display(focus_summary)

,sku_focus_flag,samples,conv_7d,avg_click_cnt,avg_distinct_skus
0,0,165420,0.087764,2.158651,10.131314
1,1,47630,0.318203,3.382658,1.843355


In [28]:
channel_type_summary = (
    sample_7d
    .groupby(['type', 'channel'], as_index=False)
    .agg(
        click_samples=('user_id', 'size'),
        conv_7d=('label_7d', 'mean'),
        avg_click_cnt=('click_cnt', 'mean'),
        avg_focus_rate=('sku_focus_flag', 'mean')
    )
    .sort_values(['type', 'click_samples'], ascending=[True, False])
)

display(channel_type_summary)

,type,channel,click_samples,conv_7d,avg_click_cnt,avg_focus_rate
0,1,app,113916,0.157669,2.740309,0.253520
4,1,wechat,8017,0.141200,2.270800,0.220157
3,1,pc,7730,0.212031,1.989521,0.181759
1,1,mobile,2352,0.257228,1.900510,0.251701
2,1,others,921,0.245385,2.187839,0.241042
5,2,app,64341,0.094621,2.116007,0.185916
9,2,wechat,9667,0.115341,2.050998,0.196752
8,2,pc,3045,0.151724,1.586535,0.132677
7,2,others,1619,0.118592,1.615195,0.083385
6,2,mobile,1442,0.176144,1.766297,0.251734


In [29]:
sample_7d.to_csv('step2_click_sample_labeled_7d.csv', index=False)
channel_summary.to_csv('step3_channel_summary.csv', index=False)
click_bucket_summary.to_csv('step3_click_bucket_summary.csv', index=False)
focus_summary.to_csv('step3_focus_summary.csv', index=False)
channel_type_summary.to_csv('step3_channel_type_summary.csv', index=False)

print('已导出。')

已导出。


In [30]:


channel_clickcnt_summary = (
    sample_7d
    .groupby(['channel', 'click_cnt_bucket'], observed=True, as_index=False)
    .agg(
        samples=('user_id', 'size'),
        users=('user_id', 'nunique'),
        conv_7d=('label_7d', 'mean'),
        avg_focus_rate=('sku_focus_flag', 'mean'),
        avg_distinct_skus=('user_day_distinct_skus', 'mean')
    )
    .sort_values(['channel', 'click_cnt_bucket'])
)

display(channel_clickcnt_summary)

,channel,click_cnt_bucket,samples,users,conv_7d,avg_focus_rate,avg_distinct_skus
0,app,1,94742,33445,0.073748,0.165840,8.416690
1,app,2,33264,19738,0.142887,0.228415,8.066198
2,app,3,16793,12475,0.200143,0.285297,7.823200
3,app,4-5,16441,12233,0.240740,0.338726,7.877076
4,app,6-10,12466,9384,0.297048,0.397962,8.396839
5,app,10+,4551,3668,0.282795,0.485827,9.847726
6,mobile,1,2355,1080,0.166030,0.195329,7.203397
7,mobile,2,707,514,0.264498,0.281471,6.239038
8,mobile,3,348,294,0.373563,0.362069,6.000000
9,mobile,4-5,238,202,0.378151,0.378151,6.058824


In [31]:
channel_clickcnt_pivot = (
    channel_clickcnt_summary
    .pivot(index='click_cnt_bucket', columns='channel', values='conv_7d')
)

display(channel_clickcnt_pivot)

channel,app,mobile,others,pc,wechat
click_cnt_bucket,,,,,
1,0.073748,0.166030,0.098819,0.124716,0.075470
2,0.142887,0.264498,0.220859,0.241114,0.148374
3,0.200143,0.373563,0.370000,0.358115,0.212792
4-5,0.240740,0.378151,0.320261,0.375857,0.238926
6-10,0.297048,0.441860,0.308824,0.392550,0.247788
10+,0.282795,0.235294,0.333333,0.377049,0.254630


In [32]:
channel_clickcnt_samples_pivot = (
    channel_clickcnt_summary
    .pivot(index='click_cnt_bucket', columns='channel', values='samples')
)

display(channel_clickcnt_samples_pivot)

channel,app,mobile,others,pc,wechat
click_cnt_bucket,,,,,
1,94742,2355,1609,6599,10004
2,33264,707,489,2082,3444
3,16793,348,200,955,1626
4-5,16441,238,153,729,1490
6-10,12466,129,68,349,904
10+,4551,17,21,61,216


In [33]:
channel_focus_summary = (
    sample_7d
    .groupby(['channel', 'sku_focus_flag'], as_index=False)
    .agg(
        samples=('user_id', 'size'),
        users=('user_id', 'nunique'),
        conv_7d=('label_7d', 'mean'),
        avg_click_cnt=('click_cnt', 'mean'),
        avg_distinct_skus=('user_day_distinct_skus', 'mean')
    )
    .sort_values(['channel', 'sku_focus_flag'])
)

display(channel_focus_summary)

,channel,sku_focus_flag,samples,users,conv_7d,avg_click_cnt,avg_distinct_skus
0,app,0,137415,25775,0.083972,2.220303,10.187076
1,app,1,40842,33045,0.306302,3.506390,1.866828
2,mobile,0,2839,875,0.164847,1.662205,8.534343
3,mobile,1,955,826,0.409424,2.406283,1.612565
4,others,0,2183,640,0.102153,1.773706,10.590930
5,others,1,357,328,0.546218,2.123249,1.308123
6,pc,0,8966,1980,0.139527,1.820544,9.377760
7,pc,1,1809,1559,0.469873,2.148701,1.656164
8,wechat,0,14017,2702,0.073982,1.931012,10.318542
9,wechat,1,3667,3134,0.329970,2.990183,1.786474


In [34]:
channel_focus_pivot = (
    channel_focus_summary
    .pivot(index='channel', columns='sku_focus_flag', values='conv_7d')
)

channel_focus_pivot = channel_focus_pivot.rename(
    columns={0: 'focus_0_conv_7d', 1: 'focus_1_conv_7d'}
)

channel_focus_pivot['focus_lift'] = (
    channel_focus_pivot['focus_1_conv_7d'] - channel_focus_pivot['focus_0_conv_7d']
)

display(channel_focus_pivot.sort_values('focus_lift', ascending=False))

sku_focus_flag,focus_0_conv_7d,focus_1_conv_7d,focus_lift
channel,,,
others,0.102153,0.546218,0.444065
pc,0.139527,0.469873,0.330346
wechat,0.073982,0.329970,0.255988
mobile,0.164847,0.409424,0.244577
app,0.083972,0.306302,0.222330


In [35]:
channel_focus_samples_pivot = (
    channel_focus_summary
    .pivot(index='channel', columns='sku_focus_flag', values='samples')
)

channel_focus_samples_pivot = channel_focus_samples_pivot.rename(
    columns={0: 'focus_0_samples', 1: 'focus_1_samples'}
)

display(channel_focus_samples_pivot)

sku_focus_flag,focus_0_samples,focus_1_samples
channel,,
app,137415,40842
mobile,2839,955
others,2183,357
pc,8966,1809
wechat,14017,3667


In [36]:
channel_clickcnt_summary.to_csv('step3_channel_clickcnt_summary.csv', index=False)
channel_focus_summary.to_csv('step3_channel_focus_summary.csv', index=False)
channel_clickcnt_pivot.to_csv('step3_channel_clickcnt_pivot.csv')
channel_focus_pivot.to_csv('step3_channel_focus_pivot.csv')

print('两张交叉表已导出。')

两张交叉表已导出。
